# Word2Vec 모델 실습하기



## 1. 실행 환경 확인 및 필수 라이브러리 설치

 Google Colab에서 Word2Vec 실습에 필요한 라이브러리를 설치합니다.

In [1]:
# Colab 또는 Jupyter 환경에서 필요한 패키지를 설치하기 위해 pip 명령을 실행합니다.
# -q 옵션은 설치 로그를 줄여 화면을 깔끔하게 유지합니다.
!pip install -q --upgrade pip

# gensim은 Word2Vec과 FastText 모델을 학습하고 사용할 때 필요한 핵심 라이브러리입니다.
!pip install -q gensim

# pandas는 CSV 파일을 읽고 데이터프레임 형태로 다룰 때 사용합니다.
!pip install -q pandas

# JPype1은 KoNLPy가 Java 기반 형태소 분석기를 사용할 때 필요한 연결 라이브러리입니다.
!pip install -q JPype1

# konlpy는 Okt 형태소 분석기를 통해 한국어 명사를 추출할 때 사용합니다.
!pip install -q konlpy

# scikit-learn은 뒤에서 단어 벡터를 2차원으로 줄여 시각화할 때 사용할 수 있습니다.
!pip install -q scikit-learn

# matplotlib은 학습 결과를 그래프로 시각화할 때 사용합니다.
!pip install -q matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 13.9 MB/s eta 0:00:00


## 2. 기본 라이브러리 불러오기 및 버전 확인

 설치된 라이브러리가 정상적으로 import 되는지 확인합니다. `gensim` import가 성공하면 이전 화면의 `No module named gensim` 오류가 해결된 것입니다.

In [2]:
# 운영체제 경로 처리, 파일 존재 여부 확인, 폴더 생성 등에 사용하는 기본 모듈입니다.
import os

# 정규표현식을 사용하여 특수문자 제거, 공백 정리 등을 수행하기 위한 기본 모듈입니다.
import re

# CSV 파일을 읽고 표 형태의 데이터프레임으로 처리하기 위한 데이터 분석 라이브러리입니다.
import pandas as pd

# gensim은 Word2Vec과 FastText 모델을 제공하는 자연어 처리 라이브러리입니다.
import gensim

# Word2Vec은 단어를 고정 길이의 숫자 벡터로 학습하는 모델 클래스입니다.
from gensim.models import Word2Vec

# FastText는 Word2Vec과 유사하지만 단어 내부의 글자 n-gram까지 학습하는 모델 클래스입니다.
from gensim.models import FastText

# 설치된 gensim 버전을 출력하여 라이브러리가 정상 설치되었는지 확인합니다.
print('gensim version:', gensim.__version__)

# 현재 작업 폴더를 출력하여 파일을 어디에서 읽고 쓰는지 확인합니다.
print('현재 작업 폴더:', os.getcwd())

gensim version: 4.4.0
현재 작업 폴더: /content


## 3. data 폴더와 news.csv 파일 준비

 `data` 폴더가 없으면 자동으로 만들고, `news.csv`가 없으면 실습용 샘플 뉴스 데이터를 자동으로 생성합니다. 따라서 별도 파일이 없어도 Word2Vec 학습을 바로 진행할 수 있습니다.

In [3]:
# 실습 데이터 파일을 저장할 폴더 경로를 지정합니다.
data_dir = './data'

# 실습 데이터 파일의 전체 경로를 지정합니다.
news_path = os.path.join(data_dir, 'news.csv')

# data 폴더가 없으면 자동으로 생성합니다.
os.makedirs(data_dir, exist_ok=True)

# news.csv 파일이 없는 경우 실습용 샘플 데이터를 자동 생성합니다.
if not os.path.exists(news_path):
    # Word2Vec이 단어 간 문맥을 조금이라도 학습할 수 있도록 여러 주제의 한국어 예문을 준비합니다.
    sample_news = [
        {'category': 7, 'news': '인공지능 기술은 자연어 처리와 컴퓨터 비전 분야에서 빠르게 발전하고 있다.'},
        {'category': 7, 'news': '머신러닝 모델은 데이터에서 패턴을 학습하고 예측 결과를 생성한다.'},
        {'category': 7, 'news': '딥러닝은 신경망 구조를 사용하여 이미지 음성 텍스트 데이터를 분석한다.'},
        {'category': 7, 'news': '자연어 처리에서는 형태소 분석 토큰화 임베딩 벡터화 과정이 중요하다.'},
        {'category': 7, 'news': 'Word2Vec 모델은 비슷한 문맥에 등장하는 단어를 가까운 벡터로 표현한다.'},
        {'category': 7, 'news': 'FastText 모델은 단어 내부의 글자 정보를 활용하여 미등록 단어 문제를 줄인다.'},
        {'category': 7, 'news': '검색 엔진은 문서와 질의 사이의 의미 유사도를 계산하여 결과를 제공한다.'},
        {'category': 7, 'news': '추천 시스템은 사용자 행동 데이터와 상품 정보를 분석하여 맞춤형 상품을 추천한다.'},
        {'category': 7, 'news': '올림픽 대회에서는 선수들이 다양한 종목에서 경쟁하고 기록을 세운다.'},
        {'category': 7, 'news': '국제 대회와 스포츠 경기는 선수 팀 감독 팬들에게 중요한 관심사이다.'},
        {'category': 7, 'news': '스마트폰 시장에서는 아이폰 갤럭시 태블릿 노트북 같은 전자기기가 경쟁한다.'},
        {'category': 7, 'news': '아이폰 사용자는 카메라 성능 배터리 화면 디자인을 중요하게 생각한다.'},
        {'category': 1, 'news': '경제 시장에서는 금리 환율 물가 주식 투자 지표가 중요하게 다루어진다.'},
        {'category': 2, 'news': '정치 뉴스에서는 정부 국회 선거 정책 법안 등이 주요 이슈로 등장한다.'},
    ]

    # 샘플 데이터를 pandas 데이터프레임으로 변환합니다.
    sample_df = pd.DataFrame(sample_news)

    # 생성한 샘플 데이터를 news.csv 파일로 저장합니다.
    sample_df.to_csv(news_path, index=False, encoding='utf-8-sig')

    # 파일 생성 완료 메시지를 출력합니다.
    print('news.csv 파일이 없어 실습용 샘플 파일을 생성했습니다:', news_path)
else:
    # 파일이 이미 있으면 기존 파일을 그대로 사용합니다.
    print('기존 news.csv 파일을 사용합니다:', news_path)

news.csv 파일이 없어 실습용 샘플 파일을 생성했습니다: ./data/news.csv


## 4. news.csv 읽기 및 컬럼 자동 확인

 `news.csv`를 읽은 뒤 실제 컬럼명을 확인합니다. 파일마다 본문 컬럼명이 `news`, `text`, `content`, `title` 등으로 다를 수 있으므로 자동으로 사용할 텍스트 컬럼을 찾습니다.

In [4]:
# news.csv 파일을 pandas 데이터프레임으로 읽어옵니다.
df_news = pd.read_csv(news_path)

# 데이터프레임의 행과 열 개수를 출력하여 파일이 정상적으로 읽혔는지 확인합니다.
print('데이터 크기:', df_news.shape)

# CSV 파일에 들어 있는 컬럼명을 출력하여 구조를 확인합니다.
print('컬럼 목록:', df_news.columns.tolist())

# 본문으로 사용할 수 있는 대표 컬럼명 후보를 순서대로 준비합니다.
text_column_candidates = ['news', 'text', 'content', 'article', '본문', '내용', 'title', '제목']

# 후보 컬럼 중 실제 데이터프레임에 존재하는 첫 번째 컬럼을 텍스트 컬럼으로 선택합니다.
text_col = next((col for col in text_column_candidates if col in df_news.columns), None)

# 후보 컬럼이 하나도 없으면 문자열 데이터가 가장 많은 컬럼을 자동으로 선택합니다.
if text_col is None:
    # object 타입은 보통 문자열 컬럼이므로 문자열 컬럼 목록을 찾습니다.
    string_columns = df_news.select_dtypes(include='object').columns.tolist()

    # 문자열 컬럼이 하나도 없으면 텍스트 분석을 진행할 수 없으므로 오류를 발생시킵니다.
    if not string_columns:
        raise ValueError('텍스트로 사용할 수 있는 문자열 컬럼이 없습니다. news.csv 컬럼을 확인하세요.')

    # 첫 번째 문자열 컬럼을 텍스트 컬럼으로 사용합니다.
    text_col = string_columns[0]

# 최종 선택된 텍스트 컬럼명을 출력합니다.
print('사용할 텍스트 컬럼:', text_col)

# 데이터 앞부분을 출력하여 내용이 정상인지 확인합니다.
df_news.head()

데이터 크기: (14, 2)
컬럼 목록: ['category', 'news']
사용할 텍스트 컬럼: news


,category,news
0,7,인공지능 기술은 자연어 처리와 컴퓨터 비전 분야에서 빠르게 발전하고 있다.
1,7,머신러닝 모델은 데이터에서 패턴을 학습하고 예측 결과를 생성한다.
2,7,딥러닝은 신경망 구조를 사용하여 이미지 음성 텍스트 데이터를 분석한다.
3,7,자연어 처리에서는 형태소 분석 토큰화 임베딩 벡터화 과정이 중요하다.
4,7,Word2Vec 모델은 비슷한 문맥에 등장하는 단어를 가까운 벡터로 표현한다.


## 5. 분석 대상 텍스트 선택 및 결합

 `category` 컬럼이 있으면 우선 카테고리 7 데이터를 선택합니다. 단, 카테고리 7 결과가 비어 있으면 전체 데이터를 사용하도록 처리하여 `명사 개수: 0` 문제를 줄입니다.

In [5]:
# category 컬럼이 있고 category 값이 7인 데이터가 하나 이상 있는지 확인합니다.
if 'category' in df_news.columns and (df_news['category'] == 7).sum() > 0:
    # category 값이 7인 행만 선택합니다.
    selected_df = df_news[df_news['category'] == 7].copy()

    # 선택 기준을 출력합니다.
    print('category == 7 데이터만 사용합니다.')
else:
    # category가 없거나 category 7 데이터가 없으면 전체 데이터를 사용합니다.
    selected_df = df_news.copy()

    # 전체 데이터를 사용한다는 안내를 출력합니다.
    print('category 7 데이터가 없거나 category 컬럼이 없어 전체 데이터를 사용합니다.')

# 선택된 데이터 개수를 출력합니다.
print('선택된 문서 수:', len(selected_df))

# 텍스트 컬럼의 결측값을 빈 문자열로 바꾼 뒤 문자열 타입으로 변환합니다.
selected_texts = selected_df[text_col].fillna('').astype(str).tolist()

# 여러 뉴스 문장을 하나의 긴 문자열로 결합합니다.
text = ' '.join(selected_texts)

# 결합된 텍스트 길이를 출력합니다.
print('결합된 텍스트 길이:', len(text))

# 결합된 텍스트 앞부분을 출력하여 실제 내용이 들어 있는지 확인합니다.
print(text[:500])

category == 7 데이터만 사용합니다.
선택된 문서 수: 12
결합된 텍스트 길이: 494
인공지능 기술은 자연어 처리와 컴퓨터 비전 분야에서 빠르게 발전하고 있다. 머신러닝 모델은 데이터에서 패턴을 학습하고 예측 결과를 생성한다. 딥러닝은 신경망 구조를 사용하여 이미지 음성 텍스트 데이터를 분석한다. 자연어 처리에서는 형태소 분석 토큰화 임베딩 벡터화 과정이 중요하다. Word2Vec 모델은 비슷한 문맥에 등장하는 단어를 가까운 벡터로 표현한다. FastText 모델은 단어 내부의 글자 정보를 활용하여 미등록 단어 문제를 줄인다. 검색 엔진은 문서와 질의 사이의 의미 유사도를 계산하여 결과를 제공한다. 추천 시스템은 사용자 행동 데이터와 상품 정보를 분석하여 맞춤형 상품을 추천한다. 올림픽 대회에서는 선수들이 다양한 종목에서 경쟁하고 기록을 세운다. 국제 대회와 스포츠 경기는 선수 팀 감독 팬들에게 중요한 관심사이다. 스마트폰 시장에서는 아이폰 갤럭시 태블릿 노트북 같은 전자기기가 경쟁한다. 아이폰 사용자는 카메라 성능 배터리 화면 디자인을 중요하게 생각한다.


## 6. 텍스트 정제

 특수문자와 불필요한 공백을 정리합니다. 정제된 텍스트가 비어 있으면 이후 단계가 실패하므로 길이를 확인합니다.

In [6]:
# 한글, 영문, 숫자, 공백을 제외한 특수문자를 공백으로 치환합니다.
cleaned_text = re.sub(r'[^가-힣a-zA-Z0-9\s]', ' ', text)

# 줄바꿈, 탭, 여러 개의 공백을 공백 한 칸으로 정리합니다.
cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()

# 정제된 텍스트 길이를 출력합니다.
print('정제된 텍스트 길이:', len(cleaned_text))

# 정제된 텍스트 앞부분을 출력하여 결과를 확인합니다.
print(cleaned_text[:500])

# 정제 후 텍스트가 비어 있으면 원본 데이터 또는 컬럼 선택에 문제가 있으므로 명확한 오류를 발생시킵니다.
if len(cleaned_text) == 0:
    raise ValueError('정제된 텍스트가 비어 있습니다. news.csv 내용과 텍스트 컬럼을 확인하세요.')

정제된 텍스트 길이: 482
인공지능 기술은 자연어 처리와 컴퓨터 비전 분야에서 빠르게 발전하고 있다 머신러닝 모델은 데이터에서 패턴을 학습하고 예측 결과를 생성한다 딥러닝은 신경망 구조를 사용하여 이미지 음성 텍스트 데이터를 분석한다 자연어 처리에서는 형태소 분석 토큰화 임베딩 벡터화 과정이 중요하다 Word2Vec 모델은 비슷한 문맥에 등장하는 단어를 가까운 벡터로 표현한다 FastText 모델은 단어 내부의 글자 정보를 활용하여 미등록 단어 문제를 줄인다 검색 엔진은 문서와 질의 사이의 의미 유사도를 계산하여 결과를 제공한다 추천 시스템은 사용자 행동 데이터와 상품 정보를 분석하여 맞춤형 상품을 추천한다 올림픽 대회에서는 선수들이 다양한 종목에서 경쟁하고 기록을 세운다 국제 대회와 스포츠 경기는 선수 팀 감독 팬들에게 중요한 관심사이다 스마트폰 시장에서는 아이폰 갤럭시 태블릿 노트북 같은 전자기기가 경쟁한다 아이폰 사용자는 카메라 성능 배터리 화면 디자인을 중요하게 생각한다


## 7. 한국어 명사 추출 함수 정의

 Okt 형태소 분석기를 우선 사용하고, Okt가 실패하거나 명사가 너무 적게 추출되면 정규표현식 기반 대체 토큰화를 사용합니다. 이 처리로 `명사 개수: 0` 상태에서도 실습이 중단되지 않습니다.

In [7]:
# 한국어 명사 추출을 안전하게 수행하는 함수를 정의합니다.
def extract_korean_tokens(input_text):
    # 최종적으로 Word2Vec 학습에 사용할 토큰 리스트를 저장할 변수를 준비합니다.
    tokens = []

    # 먼저 KoNLPy의 Okt 형태소 분석기를 사용해봅니다.
    try:
        # Okt 형태소 분석기를 불러옵니다.
        from konlpy.tag import Okt

        # Okt 객체를 생성합니다.
        okt = Okt()

        # 입력 텍스트에서 명사만 추출합니다.
        tokens = okt.nouns(input_text)

        # Okt 방식으로 추출한 토큰 개수를 출력합니다.
        print('Okt 명사 추출 개수:', len(tokens))
    except Exception as e:
        # Java, JPype, KoNLPy 문제로 Okt가 실패할 경우 오류 메시지를 출력하고 대체 방식을 사용합니다.
        print('Okt 명사 추출 실패, 대체 토큰화를 사용합니다:', repr(e))

    # Okt 결과가 너무 적으면 정규표현식 기반 대체 토큰화를 수행합니다.
    if len(tokens) < 10:
        # 한글 2글자 이상 또는 영문/숫자 2글자 이상 문자열을 단어 후보로 추출합니다.
        fallback_tokens = re.findall(r'[가-힣]{2,}|[a-zA-Z0-9]{2,}', input_text)

        # 대체 토큰화 결과를 최종 토큰으로 사용합니다.
        tokens = fallback_tokens

        # 대체 토큰화 사용 사실과 개수를 출력합니다.
        print('대체 토큰화 개수:', len(tokens))

    # 한 글자 토큰은 의미가 약한 경우가 많으므로 2글자 이상 토큰만 남깁니다.
    tokens = [token for token in tokens if len(token) >= 2]

    # 최종 토큰 리스트를 반환합니다.
    return tokens

## 8. 명사 또는 토큰 추출 실행 및 검증

Word2Vec 학습에 사용할 토큰을 추출하고 개수를 확인합니다. 토큰이 너무 적으면 실습용 기본 토큰을 추가하여 모델 학습 오류를 방지합니다.

In [8]:
# 정제된 텍스트에서 한국어 명사 또는 대체 토큰을 추출합니다.
noun2more = extract_korean_tokens(cleaned_text)

# Word2Vec 학습에 사용할 토큰이 너무 적으면 샘플 토큰을 추가하여 학습 실패를 방지합니다.
if len(noun2more) < 20:
    # 실습용 기본 토큰을 여러 번 반복하여 최소 학습량을 확보합니다.
    fallback_training_tokens = ['인공지능', '자연어', '처리', '모델', '학습', '데이터', '분석', '벡터', '임베딩', '유사도', '대회', '올림픽', '아이폰', '스마트폰'] * 5

    # 기존 토큰 뒤에 실습용 기본 토큰을 추가합니다.
    noun2more.extend(fallback_training_tokens)

    # 토큰이 부족해 보강했다는 메시지를 출력합니다.
    print('토큰 수가 부족하여 실습용 기본 토큰을 추가했습니다.')

# 추출된 토큰 앞부분을 출력합니다.
print('토큰 샘플:', noun2more[:50])

# 전체 토큰 개수를 출력합니다.
print('토큰 개수:', len(noun2more))

Okt 명사 추출 개수: 102
토큰 샘플: ['인공', '지능', '기술', '자연어', '처리', '컴퓨터', '분야', '발전', '머신', '러닝', '모델', '데이터', '패턴', '학습', '예측', '결과', '러닝', '신경망', '구조', '사용', '이미지', '음성', '텍스트', '데이터', '분석', '자연어', '처리', '형태소', '분석', '토큰', '임베딩', '벡터', '과정', '모델', '문맥', '등장', '단어', '벡터', '표현', '모델', '단어', '내부', '글자', '정보', '활용', '미등록', '단어', '문제', '검색', '엔진']
토큰 개수: 98


## 9. Word2Vec 입력 문장 구성

Word2Vec은 `[[단어1, 단어2, ...], [단어1, 단어2, ...]]`처럼 문장 단위 리스트를 입력으로 받습니다.  긴 토큰 리스트를 일정 길이의 여러 문장으로 나누어 학습 품질을 높입니다.

In [9]:
# 한 문장에 넣을 토큰 개수를 지정합니다.
sentence_size = 20

# 토큰 리스트를 sentence_size 단위로 잘라 여러 문장 리스트로 구성합니다.
sentences = [noun2more[i:i + sentence_size] for i in range(0, len(noun2more), sentence_size)]

# 길이가 2 이상인 문장만 남겨 Word2Vec 학습 안정성을 높입니다.
sentences = [sentence for sentence in sentences if len(sentence) >= 2]

# 생성된 문장 개수를 출력합니다.
print('학습 문장 개수:', len(sentences))

# 첫 번째 문장을 출력하여 입력 형태를 확인합니다.
print('첫 번째 학습 문장:', sentences[0])

학습 문장 개수: 5
첫 번째 학습 문장: ['인공', '지능', '기술', '자연어', '처리', '컴퓨터', '분야', '발전', '머신', '러닝', '모델', '데이터', '패턴', '학습', '예측', '결과', '러닝', '신경망', '구조', '사용']


## 10. Word2Vec 모델 학습

 준비된 토큰 문장으로 Word2Vec 모델을 학습합니다. 데이터가 적은 실습 환경에서는 `min_count=1`로 설정해야 단어가 제거되어 검색 오류가 나는 문제를 줄일 수 있습니다.

In [10]:
# Word2Vec 모델을 학습합니다.
model = Word2Vec(
    # Word2Vec 학습에 사용할 문장 리스트를 입력합니다.
    sentences=sentences,
    # sg=1은 Skip-gram 방식으로, 중심 단어를 기준으로 주변 단어를 예측합니다.
    sg=1,
    # vector_size는 단어 하나를 몇 차원 벡터로 표현할지 정합니다.
    vector_size=100,
    # window는 중심 단어 주변 몇 개 단어를 문맥으로 볼지 정합니다.
    window=3,
    # min_count=1은 한 번만 등장한 단어도 학습에 포함한다는 의미입니다.
    min_count=1,
    # workers는 학습에 사용할 CPU 스레드 수입니다.
    workers=2,
    # epochs는 전체 데이터를 몇 번 반복 학습할지 정합니다.
    epochs=100,
    # seed는 실행할 때마다 비슷한 결과가 나오도록 난수값을 고정합니다.
    seed=42
)

# 학습된 단어 사전 크기를 출력합니다.
print('Word2Vec 단어 사전 크기:', len(model.wv.index_to_key))

# 학습된 단어 일부를 출력합니다.
print('단어 사전 샘플:', model.wv.index_to_key[:30])

Word2Vec 단어 사전 크기: 77
단어 사전 샘플: ['단어', '분석', '데이터', '모델', '아이폰', '경쟁', '선수', '대회', '상품', '사용자', '추천', '정보', '벡터', '결과', '러닝', '처리', '자연어', '생각', '디자인', '화면', '배터리', '성능', '카메라', '전자기기', '노트북', '태블릿', '갤럭시', '시장', '스마트폰', '관심사']


## 11. 안전한 유사 단어 검색 함수 정의

 검색 단어가 모델 사전에 없을 때 발생하는 `KeyError`를 방지합니다. 검색 단어가 없으면 사용 가능한 단어 목록을 보여주고, 실습이 중단되지 않도록 처리합니다.

In [11]:
# Word2Vec 또는 FastText 모델에서 유사 단어를 안전하게 검색하는 함수를 정의합니다.
def safe_most_similar(trained_model, word, topn=10):
    # 입력 단어가 모델의 단어 사전에 존재하는지 확인합니다.
    if word in trained_model.wv.key_to_index:
        # 단어가 존재하면 most_similar 함수로 유사 단어를 계산합니다.
        return trained_model.wv.most_similar(word, topn=topn)

    # 입력 단어가 없으면 안내 메시지를 출력합니다.
    print(f'[{word}] 단어가 모델 사전에 없습니다.')

    # 대신 사용할 수 있는 단어 사전 샘플을 출력합니다.
    print('사용 가능한 단어 예시:', trained_model.wv.index_to_key[:30])

    # 오류 대신 빈 리스트를 반환하여 다음 코드 실행이 중단되지 않게 합니다.
    return []

# 두 단어 사이의 유사도를 안전하게 계산하는 함수를 정의합니다.
def safe_similarity(trained_model, word1, word2):
    # 첫 번째 단어가 사전에 없으면 안내하고 None을 반환합니다.
    if word1 not in trained_model.wv.key_to_index:
        print(f'[{word1}] 단어가 모델 사전에 없습니다.')
        return None

    # 두 번째 단어가 사전에 없으면 안내하고 None을 반환합니다.
    if word2 not in trained_model.wv.key_to_index:
        print(f'[{word2}] 단어가 모델 사전에 없습니다.')
        return None

    # 두 단어가 모두 사전에 있으면 코사인 유사도를 계산하여 반환합니다.
    return trained_model.wv.similarity(word1, word2)

## 12. Word2Vec 유사 단어 검색

 `대회`와 가까운 단어를 검색합니다. 사전에 없는 단어를 검색해도 오류가 발생하지 않도록 `safe_most_similar()` 함수를 사용합니다.

In [12]:
# '대회'와 의미적으로 가까운 단어들을 안전하게 검색합니다.
similar_words = safe_most_similar(model, '대회', topn=10)

# 검색 결과를 출력합니다.
print(similar_words)

[('데이터', 0.8950121998786926), ('선수', 0.8785086870193481), ('추천', 0.8765431046485901), ('모델', 0.8750178813934326), ('올림픽', 0.8741441965103149), ('러닝', 0.8729442358016968), ('분석', 0.8706234693527222), ('단어', 0.8703733086585999), ('사이', 0.8676942586898804), ('결과', 0.8671042919158936)]


## 13. Word2Vec 단어 유사도 계산

 `대회`와 `올림픽` 사이의 코사인 유사도를 계산합니다. 두 단어 중 하나가 사전에 없으면 오류 대신 안내 메시지를 출력합니다.

In [13]:
# '대회'와 '올림픽' 사이의 코사인 유사도를 안전하게 계산합니다.
similarity_score = safe_similarity(model, '대회', '올림픽')

# 계산 결과를 출력합니다.
print('대회-올림픽 유사도:', similarity_score)

대회-올림픽 유사도: 0.87414414


## 14. Word2Vec 다른 단어 검색

 `아이폰`과 가까운 단어를 검색합니다. 기존 코드에서 `아이폰`이 사전에 없으면 오류가 발생할 수 있었으므로 안전 검색 함수를 적용했습니다.

In [14]:
# '아이폰'과 의미적으로 가까운 단어들을 안전하게 검색합니다.
iphone_similar_words = safe_most_similar(model, '아이폰', topn=10)

# 검색 결과를 출력합니다.
print(iphone_similar_words)

[('모델', 0.8450037240982056), ('문제', 0.8371212482452393), ('분석', 0.8358805179595947), ('머신', 0.8341004252433777), ('문맥', 0.8335747122764587), ('단어', 0.8324171304702759), ('갤럭시', 0.8298584222793579), ('임베딩', 0.8279791474342346), ('형태소', 0.8275155425071716), ('러닝', 0.8273466229438782)]


## 15. FastText 모델 학습

 같은 데이터로 FastText 모델을 학습합니다. FastText는 단어 내부의 글자 조각 정보를 활용하므로 Word2Vec보다 작은 데이터에서도 일부 단어 처리에 더 유연할 수 있습니다.

In [15]:
# FastText 모델을 학습합니다.
f_model = FastText(
    # FastText 학습에 사용할 문장 리스트를 입력합니다.
    sentences=sentences,
    # sg=1은 Skip-gram 방식을 사용한다는 의미입니다.
    sg=1,
    # vector_size는 단어 벡터의 차원 수입니다.
    vector_size=100,
    # window는 중심 단어 주변 문맥 범위입니다.
    window=3,
    # min_count=1은 실습 데이터가 적어도 단어가 제거되지 않게 합니다.
    min_count=1,
    # workers는 학습에 사용할 CPU 스레드 수입니다.
    workers=2,
    # epochs는 전체 데이터를 반복 학습하는 횟수입니다.
    epochs=100,
    # seed는 결과 재현성을 위한 난수 고정값입니다.
    seed=42
)

# FastText 단어 사전 크기를 출력합니다.
print('FastText 단어 사전 크기:', len(f_model.wv.index_to_key))

FastText 단어 사전 크기: 77


## 16. FastText 유사 단어 검색 및 유사도 계산

 FastText 모델에서 `대회`, `올림픽`, `아이폰` 관련 결과를 확인합니다. Word2Vec 결과와 비교하여 모델 특성을 이해할 수 있습니다.

In [16]:
# FastText 모델에서 '대회'와 가까운 단어를 안전하게 검색합니다.
print('FastText 대회 유사 단어:', safe_most_similar(f_model, '대회', topn=10))

# FastText 모델에서 '대회'와 '올림픽'의 유사도를 안전하게 계산합니다.
print('FastText 대회-올림픽 유사도:', safe_similarity(f_model, '대회', '올림픽'))

# FastText 모델에서 '아이폰'과 가까운 단어를 안전하게 검색합니다.
print('FastText 아이폰 유사 단어:', safe_most_similar(f_model, '아이폰', topn=10))

FastText 대회 유사 단어: [('자연어', 0.9645913243293762), ('데이터', 0.9601401090621948), ('아이폰', 0.957707941532135), ('컴퓨터', 0.9564915895462036), ('분석', 0.9532400369644165), ('올림픽', 0.952755331993103), ('형태소', 0.9516468048095703), ('갤럭시', 0.9504773616790771), ('단어', 0.9483096599578857), ('모델', 0.947443425655365)]
FastText 대회-올림픽 유사도: 0.95275533
FastText 아이폰 유사 단어: [('자연어', 0.9730395674705505), ('데이터', 0.9676382541656494), ('컴퓨터', 0.9593073129653931), ('단어', 0.9592666029930115), ('형태소', 0.9579437971115112), ('대회', 0.957707941532135), ('임베딩', 0.9575343728065491), ('배터리', 0.9572356939315796), ('갤럭시', 0.9549024105072021), ('질의', 0.9545637965202332)]


## 17. Word2Vec 모델 저장 및 다시 불러오기

 학습한 Word2Vec 모델을 파일로 저장하고 다시 불러옵니다. 모델을 저장하면 다음 실습에서 다시 학습하지 않고 사용할 수 있습니다.

In [17]:
# 학습된 모델을 저장할 폴더를 생성합니다.
os.makedirs('./models', exist_ok=True)

# Word2Vec 모델 저장 경로를 지정합니다.
word2vec_model_path = './models/korean_word2vec_practice.model'

# 학습된 Word2Vec 모델을 파일로 저장합니다.
model.save(word2vec_model_path)

# 저장된 Word2Vec 모델을 다시 불러옵니다.
loaded_model = Word2Vec.load(word2vec_model_path)

# 다시 불러온 모델의 단어 사전 크기를 출력하여 정상 저장 여부를 확인합니다.
print('다시 불러온 모델 단어 사전 크기:', len(loaded_model.wv.index_to_key))

다시 불러온 모델 단어 사전 크기: 77
